# Практическое занятие 4. Создание небольшого текстового корпуса на узбекском языке и вычисление статистических характеристик корпуса

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkb40-svg/big_data_6semestr/blob/main/notebooks/p04.ipynb)

**Цели занятия:**

- Собрать корпус из текстовых файлов и метаданных в удобную структуру: словарь и `DataFrame`
- Вычислить основные характеристики корпуса: символы, предложения, токены, типы, TTR, гапаксы, средние длины
- Сравнить характеристики по категориям текстов и сохранить корпус в форматах CSV и JSON Lines

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/mkb40-svg/big_data_6semestr.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Что такое корпус

**Корпус** — это собрание текстов, отобранных по понятным правилам и снабжённых **метаданными** (автор, дата, жанр, источник). Именно метаданные отличают корпус от «папки с файлами»: благодаря им можно сравнивать, например, язык спортивных и научных новостей.

Крупные корпуса — Британский национальный корпус, Национальный корпус русского языка, трибанки Universal Dependencies — содержат миллионы и миллиарды словоупотреблений. Принципы работы с ними те же, что и с нашим учебным корпусом из 16 новостей; меняются только объём и инструменты.

| Файл | Содержимое |
|---|---|
| `data/corpus_uz/uz_01.txt … uz_16.txt` | новостные тексты на узбекском (латиница); первая строка — заголовок |
| `data/corpus_uz/meta.csv` | file, title, category, date, author, apostrophe_style, url |

## Чтение корпуса в Python

```python
import glob, os
import pandas as pd

corpus = {}
for path in sorted(glob.glob("data/corpus_uz/uz_*.txt")):
    with open(path, encoding="utf-8") as f:      # всегда указывайте кодировку
        corpus[os.path.basename(path)] = f.read()

meta = pd.read_csv("data/corpus_uz/meta.csv")
meta["text"] = meta["file"].map(corpus)           # корпус = метаданные + тексты
```

> **Внимание: Порядок файлов**  
> `os.listdir` и `glob.glob` **не гарантируют** порядок файлов (он зависит от файловой системы). Чтобы результаты были воспроизводимыми, всегда сортируйте список: `sorted(...)`.

## Ловушка узбекской латиницы: апострофы

В официальной латинице буквы **oʻ** и **gʻ** пишутся со знаком ʻ (U+02BB), а разделительный знак *tutuq belgisi* — ʼ (U+02BC). На практике в текстах встречаются ASCII `'`, типографские `‘ ’` и даже обратный апостроф. Файлы нашего корпуса специально записаны **разными** способами:

| Вариант | Пример | `re.findall(r"\w+", ...)` |
|---|---|---|
| канонический | oʻzbek, maʼlumot | `['oʻzbek']` |
| ASCII | o'zbek, ma'lumot | `['o', 'zbek']` — слово разорвано! |
| типографский | o‘zbek, ma’lumot | `['o', 'zbek']` |

Знаки ʻ и ʼ относятся к категории Unicode **Lm** (буква-модификатор), поэтому `\w` считает их частью слова. Значит, достаточно привести все варианты к каноническим — и простая регулярка `\w+` будет правильно делить текст на слова:

```python
import re
def normalize(text):
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

tokens = re.findall(r"\w+", normalize(text).lower())
```

Подробно нормализацию разберём в П12, а здесь будем пользоваться готовой функцией.

## Статистические характеристики корпуса

| Величина | Как считать | Что показывает |
|---|---|---|
| символы | `len(text)` | объём «сырого» текста |
| предложения | деление по `. ! ?` и переводам строк | синтаксическую «плотность» |
| **токены** (словоупотребления) | `len(tokens)` | объём корпуса в словах |
| **типы** (словоформы) | `len(set(tokens))` | размер словаря |
| **TTR** (type-token ratio) | типы / токены | лексическое разнообразие |
| **гапаксы** (hapax legomena) | слова с частотой 1 | «длинный хвост» словаря |
| средняя длина слова | `sum(map(len, tokens)) / len(tokens)` | у агглютинативных языков она больше |
| средняя длина предложения | токены / предложения | сложность синтаксиса |

```python
from collections import Counter
freq = Counter(tokens)
n_tokens, n_types = len(tokens), len(freq)
ttr = n_types / n_tokens
hapax = [w for w, c in freq.items() if c == 1]
```

Узбекский язык **агглютинативный**: к корню присоединяются цепочки суффиксов (*kitob → kitoblar → kitoblarimizdan*). Поэтому одна лексема даёт десятки словоформ: словарь растёт быстрее, TTR и доля гапаксов выше, чем, например, в английском тексте того же объёма.

> **Заметка: TTR зависит от длины текста**  
> Чем длиннее текст, тем чаще повторяются слова и тем ниже TTR (рост словаря замедляется — закон Хипса). Поэтому TTR сравнивают только у текстов **одинаковой длины** или считают на выборках фиксированного размера.

## Группировка по метаданным

```python
df.groupby("category")["n_tokens"].sum()          # токены по категориям
```

TTR категории нужно считать по **объединённым** текстам категории (типы всех её текстов / токены всех её текстов), а не как среднее TTR документов.

## Сохранение корпуса

- **CSV** — таблица, удобна для Excel и pandas, но длинные тексты с переводами строк и кавычками экранируются.
- **JSON Lines** (`.jsonl`) — одна строка = один JSON-документ. Стандарт для больших корпусов (HuggingFace datasets, Spark): файл можно читать построчно, не загружая целиком.

```python
df.to_csv("corpus.csv", index=False)
df.to_json("corpus.jsonl", orient="records", lines=True, force_ascii=False)
back = pd.read_json("corpus.jsonl", lines=True)
```

> **Совет: force_ascii=False**  
> Без этого параметра каждая буква ʻ превратится в шестисимвольную последовательность `\u02bb`: файл станет больше и нечитаемым для человека.

## Задание 1. Чтение одного текста

Откройте файл `data/corpus_uz/uz_04.txt` (кодировка UTF-8) и прочитайте его целиком в переменную `text`. Первая строка файла — заголовок новости.

Создайте переменные:
- `title` — первая строка без пробелов по краям;
- `body` — весь остальной текст без пробелов и пустых строк по краям (`.strip()`);
- `n_chars` — число символов в `body`;
- `n_paragraphs` — число непустых строк (абзацев) в `body`.

<details><summary>💡 Подсказка</summary>

`f.read()` читает файл целиком. Строки без первой: `"\n".join(lines[1:])`. Непустые строки: `[l for l in body.split("\n") if l.strip()]`.

</details>

In [ ]:
with open("data/corpus_uz/uz_04.txt", encoding="utf-8") as f:
    text = ...

lines = text.split("\n")
title = ...
body = ...
n_chars = ...
n_paragraphs = ...

print(title)
print("Символов:", n_chars, "абзацев:", n_paragraphs)

## Задание 2. Корпус как словарь

Прочитайте все 16 файлов `uz_*.txt` из папки `data/corpus_uz/` в словарь `corpus = {имя_файла: текст}`. Ключ — имя файла **без пути** (например, `"uz_01.txt"`), значение — полный текст.

Ключи должны идти по порядку `uz_01.txt … uz_16.txt` — используйте `sorted(glob.glob(...))`.

Посчитайте `total_chars` — суммарное число символов во всех текстах — и выведите на экран число документов.

<details><summary>💡 Подсказка</summary>

`glob.glob("data/corpus_uz/uz_*.txt")` возвращает пути в произвольном порядке — оберните в `sorted()`. Имя файла: `os.path.basename(path)`. Сумма длин: `sum(len(t) for t in corpus.values())`.

</details>

In [ ]:
import glob, os

paths = ...          # отсортированный список путей к файлам uz_*.txt
corpus = {}
for path in paths:
    name = ...       # имя файла без папки
    with open(path, encoding="utf-8") as f:
        corpus[name] = ...

total_chars = ...
print("Документов:", len(corpus))
print("Символов:", total_chars)

## Задание 3. Корпус с метаданными в DataFrame

Метаданные корпуса лежат в `data/corpus_uz/meta.csv` (столбцы file, title, category, date, author, apostrophe_style, url).

1. Прочитайте meta.csv в `df`.
2. Добавьте столбец `text` — текст файла, указанного в столбце `file` (файлы лежат в той же папке).
3. Добавьте столбец `n_chars` — длина текста в символах.
4. Сохраните в `cat_counts` число документов в каждой категории (`value_counts`).

<details><summary>💡 Подсказка</summary>

`df["file"].apply(read_text)` применит функцию к каждому имени файла; длина строк — `df["text"].str.len()`.

</details>

In [ ]:
import pandas as pd

df = ...

def read_text(name):
    with open("data/corpus_uz/" + name, encoding="utf-8") as f:
        return f.read()

df["text"] = ...
df["n_chars"] = ...
cat_counts = ...

print(cat_counts)
df[["file", "title", "category", "n_chars"]]

## Задание 4. Токены, типы, TTR и гапаксы

В подготовленном коде загружена таблица `df` (столбцы file, title, category, text) и функция `normalize(text)`, которая приводит все варианты апострофа к каноническим ʻ и ʼ.

Токенизируйте **весь корпус** (документы по порядку) так: `re.findall(r"\w+", normalize(text).lower())` и вычислите:
- `tokens` — список всех токенов корпуса;
- `n_tokens` и `n_types` — число токенов и типов (уникальных словоформ);
- `ttr` — type-token ratio = `n_types / n_tokens`;
- `hapax` — **множество** слов, встретившихся ровно один раз, и `hapax_share` — их доля среди типов;
- `n_tokens_raw` — число токенов при такой же токенизации, но **без** `normalize`. Почему оно больше?

<details><summary>💡 Подсказка</summary>

Частоты даёт `Counter(tokens)`; гапаксы — `{w for w, c in freq.items() if c == 1}`. Для `n_tokens_raw` повторите токенизацию для `text.lower()` без `normalize`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]

In [ ]:
import re
from collections import Counter

tokens = []
for text in df["text"]:
    tokens += ...

freq = Counter(tokens)
n_tokens = ...
n_types = ...
ttr = ...
hapax = ...
hapax_share = ...

n_tokens_raw = ...

print(f"Токенов: {n_tokens}, типов: {n_types}, TTR = {ttr}")
print("Гапаксов:", len(hapax), "доля:", hapax_share)
print("Без нормализации:", n_tokens_raw)

## Задание 5. Характеристики каждого документа

Добавьте в таблицу `df` столбцы с характеристиками **каждого** документа:

- `n_tokens`, `n_types`, `ttr` — как в прошлом задании, но для одного текста;
- `n_sents` — число предложений. Делите текст так: `re.split(r"(?<=[.!?])\s+|\n+", text.strip())` и отбрасывайте пустые части (заголовок считается отдельным предложением);
- `avg_word_len` — средняя длина токена (после нормализации) в символах;
- `avg_sent_len` — среднее число токенов в предложении: `n_tokens / n_sents`.

Строки `df` не удаляйте. Выведите таблицу, отсортированную по `ttr`: у каких текстов словарь разнообразнее?

<details><summary>💡 Подсказка</summary>

Предложения: `[s for s in re.split(r"(?<=[.!?])\s+|\n+", text.strip()) if s.strip()]`. Средняя длина слова: `sum(len(t) for t in toks) / len(toks)`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]

In [ ]:
def doc_stats(text):
    toks = re.findall(r"\w+", normalize(text).lower())
    sents = ...
    return pd.Series({
        "n_tokens": len(toks),
        "n_types": ...,
        "ttr": ...,
        "n_sents": ...,
        "avg_word_len": ...,
        "avg_sent_len": ...,
    })

stats = df["text"].apply(doc_stats)
df = pd.concat([df, stats], axis=1)
df[["file", "category", "n_tokens", "n_types", "ttr", "n_sents", "avg_word_len", "avg_sent_len"]].sort_values("ttr")

## Задание 6. Статистика по категориям

В `df` уже есть столбец `tokens` — список токенов каждого документа.

Постройте таблицу `cat_stats` (индекс — `category`) со столбцами:
- `docs` — число документов категории;
- `tokens` — общее число токенов;
- `types` — число **уникальных** слов во всех текстах категории вместе;
- `ttr` — `types / tokens` (TTR объединённого текста категории, а **не** среднее TTR документов!).

Отсортируйте таблицу по `tokens` по убыванию и постройте столбчатую диаграмму числа токенов по категориям.

<details><summary>💡 Подсказка</summary>

Все токены группы: `[t for toks in group["tokens"] for t in toks]`. Сортировка: `cat_stats.sort_values("tokens", ascending=False)`. Диаграмма: `cat_stats["tokens"].plot(kind="bar")` и `plt.show()`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]

df["tokens"] = df["text"].apply(lambda t: re.findall(r"\w+", normalize(t).lower()))

In [ ]:
import matplotlib.pyplot as plt

rows = []
for cat, group in df.groupby("category"):
    all_toks = ...      # все токены документов категории одним списком
    rows.append({"category": cat, "docs": ..., "tokens": ..., "types": ...})

cat_stats = pd.DataFrame(rows).set_index("category")
cat_stats["ttr"] = ...
cat_stats = ...

# диаграмма
...
cat_stats

## Задание 7. Сохранение корпуса: CSV и JSON Lines

Сохраните подготовленный корпус для дальнейшей работы.

1. Замените столбец `text` в `df` его нормализованной версией (`normalize`) и добавьте столбец `n_tokens` (токены — как раньше, `\w+` после `.lower()`).
2. Сохраните `df` в `corpus_uz.csv` (без индекса).
3. Сохраните `df` в `corpus_uz.jsonl` в формате JSON Lines: `orient="records", lines=True, force_ascii=False`.
4. Прочитайте JSONL обратно в `back` (`pd.read_json(..., lines=True)`) и выведите размеры обоих файлов в байтах (`os.path.getsize`).

<details><summary>💡 Подсказка</summary>

`df.to_json("corpus_uz.jsonl", orient="records", lines=True, force_ascii=False)`; число токенов: `df["text"].str.lower().str.findall(r"\w+").str.len()`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]

import os
for _f in ["corpus_uz.csv", "corpus_uz.jsonl"]:
    if os.path.exists(_f):
        os.remove(_f)

In [ ]:
import os

df["text"] = ...
df["n_tokens"] = ...

# 1) CSV
...
# 2) JSON Lines
...

back = ...
print("CSV:", os.path.getsize("corpus_uz.csv"), "байт")
print("JSONL:", os.path.getsize("corpus_uz.jsonl"), "байт")
back.head(3)

## Самостоятельная работа

**Тема для самостоятельной работы: «Перспективы применения технологий Big Data в компьютерной лингвистике».**

1. Соберите собственный мини-корпус из 20–30 узбекских текстов одного жанра (новости, отзывы, посты) с файлом метаданных.
2. Посчитайте для него все характеристики из занятия и сравните с учебным корпусом.
3. Напишите короткое эссе (1–2 страницы): какие задачи компьютерной лингвистики требуют корпусов в миллиарды токенов и какие технологии Big Data (HDFS, Spark, JSON Lines, потоковая обработка) для этого нужны.